# 02 - Feature engineering and model development
The production training script is `python -m src.train` (with MLflow). This notebook walks through the same steps interactively.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make `src` importable from notebooks/
warnings.filterwarnings("ignore")
import pandas as pd
from IPython.display import Image, display
from sklearn.model_selection import train_test_split
from src import modeling
from src.config import FEATURES, TARGET, RANDOM_STATE
from src.data import load_clean
df = load_clean()
X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES], df[TARGET], test_size=0.2, stratify=df[TARGET], random_state=RANDOM_STATE)
X_train.shape, X_test.shape

## Feature engineering
`pct_max_hr` = thalach / (220 - age). Numerics are median-imputed + standardised; categoricals are mode-imputed + one-hot encoded.

In [ ]:
from src.features import add_clinical_features
add_clinical_features(X_train)[['age','thalach','pct_max_hr']].head()

## Tune and compare three models (5-fold stratified CV, ROC-AUC)

In [ ]:
rows, fitted = {}, {}
for name, (est, grid) in modeling.candidate_models().items():
    search = modeling.tune(name, est, grid, X_train, y_train)
    best = search.best_estimator_
    rows[name] = {**modeling.cross_validated_metrics(best, X_train, y_train),
                  **modeling.holdout_metrics(best, X_test, y_test)}
    fitted[name] = best
    print(name, search.best_params_)
results = pd.DataFrame(rows).T
results[['cv_accuracy_mean','cv_precision_mean','cv_recall_mean','cv_roc_auc_mean','test_roc_auc']].round(3)

In [ ]:
modeling.plot_model_comparison(results)

In [ ]:
best_name = results['cv_roc_auc_mean'].idxmax()
print('Selected:', best_name)
modeling.plot_roc(fitted[best_name], X_test, y_test, best_name)

In [ ]:
modeling.plot_confusion(fitted[best_name], X_test, y_test, best_name)

In [ ]:
modeling.plot_feature_importance(fitted[best_name], 'Top features - ' + best_name)

## Model selection notes
*(Fill in: why the selected model won, recall vs precision trade-off for a screening use case.)*